In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import pickle
import lightgbm as lgb
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import warnings
import re
warnings.filterwarnings('ignore')

print("All libraries loaded!")

All libraries loaded!


In [8]:
# Load enriched data
df = pd.read_csv(r'C:\Users\HP\Documents\Projects\predictive-Maintenance\data\processed\week2_enriched.csv')

# Drop non-numeric columns
drop_cols = ['Product ID', 'Type', 'timestamp']
df_model  = df.drop(columns=[c for c in drop_cols 
                              if c in df.columns])

# Features and target
X = df_model.drop(columns=['Machine failure'])
y = df_model['Machine failure']

# Clean column names
def clean_col_name(col):
    col = re.sub(r'[\[\]\(\)\{\}<>]', '', col)
    col = re.sub(r'[^A-Za-z0-9_]', '_', col)
    col = re.sub(r'\+', '', col)
    return col.strip('_')

X.columns = [clean_col_name(c) for c in X.columns]

# Load saved model
with open(r'C:\predictive_maintenance\models\lgbm_model.pkl', 'rb') as f:
    model = pickle.load(f)

print("Data and model loaded!")
print("Data shape:", df.shape)
print("Model type:", type(model).__name__)

Data and model loaded!
Data shape: (9991, 83)
Model type: LGBMClassifier


In [9]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Apply SMOTE on training data only
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# Retrain model with clean column names
model = lgb.LGBMClassifier(
    n_estimators     = 500,
    learning_rate    = 0.05,
    max_depth        = 6,
    num_leaves       = 31,
    min_child_samples= 20,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    class_weight     = 'balanced',
    random_state     = 42,
    verbose          = -1
)

model.fit(X_train_sm, y_train_sm)
print("Model retrained successfully!")
print(f"Training rows after SMOTE: {X_train_sm.shape[0]}")

Model retrained successfully!
Training rows after SMOTE: 15442
